# 第 2 章教学讲义：GPU 硬件模型与执行层次

这个 notebook 用来把 GPU 的执行层次、内存层次、计算单元和 tile 设计串成一个可教学的判断框架。

## 1. 执行层次

- Thread / Warp / Block / Grid 不是随便并列的名词
- 它们决定了同步边界、共享资源边界和通信成本
- 线程越多，不一定吞吐越高，关键是协同是否高效

In [ ]:
layers = ["Thread", "Warp", "Block", "Grid"]
for idx, layer in enumerate(layers, start=1):
    print(f"Level {idx}: {layer}")

## 2. 内存层次

- Register 最快，但最小、最私有
- Shared Memory 适合 block 内复用
- L2 共享范围更大，但代价更高
- Global / HBM 容量最大，但最慢

判断的关键不是“有没有这一层”，而是“数据应该落在哪一层”。

## 3. 计算单元

- CUDA Core 适合通用算术逻辑
- Tensor Core 适合矩阵乘加
- 形状、布局和 tile 不匹配时，专用单元的优势会被浪费掉

In [ ]:
def arithmetic_intensity(flops, bytes_accessed):
    return flops / bytes_accessed

M = N = K = 1024
flops = 2 * M * N * K
bytes_accessed = (M*K + K*N + M*N) * 4
print(f"AI = {arithmetic_intensity(flops, bytes_accessed):.2f} FLOP/Byte")

## 4. 为什么需要 tile

- 把数据复用到更快的层次
- 控制并行粒度
- 对齐硬件执行边界
- 把大问题拆成更容易验证和调优的小问题

## 5. 本章结论

读完这章，读者应该能先判断一个 kernel 更可能卡在哪一层，再决定是改执行粒度、改内存层次，还是改 tile / layout。

## 6. 一个小例子：为什么 tile 会改变瓶颈

假设一个算子反复读取同一块输入数据：

- 不分块时，这些数据会一次次从更慢的层次搬运
- 分块后，同一块数据可以先放到更快的片上层里复用

因此，tile 的价值不是“切得更细”，而是“让热点数据少走慢路径”。

In [ ]:
def classify_ai(flops, bytes_accessed, peak_compute_tflops=312.0, peak_memory_gbs=2000.0):
    ai = flops / bytes_accessed
    roofline_tflops = ai * peak_memory_gbs / 1e3
    return ai, ("compute-bound" if roofline_tflops >= peak_compute_tflops else "memory-bound")

ai, bottleneck = classify_ai(2 * 1024**3, (3 * 1024**2) * 4)
print(f"AI = {ai:.2f} FLOP/Byte")
print(f"Bottleneck = {bottleneck}")

## 7. 一个实用结论

如果你只能记住一件事：

- 先判断算子主要受执行层、内存层还是计算单元限制
- 再决定要不要改 tile、layout 或计算映射
- 不要一上来就假设问题只是“代码写得不够快”